# Inference Pipeline Demo

This notebook demonstrates the full dice inference pipeline:

1. dice detection
2. crop extraction
3. dice face classification
4. visualization of final predictions
5. optional batch inference on a folder
6. export of results to CSV

In [ ]:
from pathlib import Path
import csv

import numpy as np
import keras
import keras_cv
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from project_config import (
    BOUNDING_BOX_FORMAT,
    DETECTOR_TARGET_SIZE,
    CLASSIFIER_IMG_SIZE,
    CLASS_NAMES,
    DETECTOR_MODEL_PATH,
    CLASSIFIER_MODEL_PATH,
    DETECTOR_CONFIDENCE_THRESHOLD,
    DETECTOR_IOU_THRESHOLD,
    CROP_MARGIN,
    INFERENCE_IMAGE_PATH,
    INFERENCE_RESULTS_CSV,
)

from utils.pipeline import (
    load_image_bgr,
    bgr_to_rgb,
    preprocess_for_detector,
    map_box_to_original,
    expand_box,
    crop_from_box,
    preprocess_crop_for_classifier,
)

In [ ]:
assert DETECTOR_MODEL_PATH.exists(), f"Missing detector model: {DETECTOR_MODEL_PATH}"
assert CLASSIFIER_MODEL_PATH.exists(), f"Missing classifier model: {CLASSIFIER_MODEL_PATH}"

print("DETECTOR_MODEL_PATH:", DETECTOR_MODEL_PATH)
print("CLASSIFIER_MODEL_PATH:", CLASSIFIER_MODEL_PATH)

In [ ]:
detector = keras.models.load_model(DETECTOR_MODEL_PATH)
detector.prediction_decoder = keras_cv.layers.NonMaxSuppression(
    bounding_box_format=BOUNDING_BOX_FORMAT,
    from_logits=False,
    confidence_threshold=DETECTOR_CONFIDENCE_THRESHOLD,
    iou_threshold=DETECTOR_IOU_THRESHOLD,
)

classifier = keras.models.load_model(CLASSIFIER_MODEL_PATH)

print("Models loaded successfully.")

In [ ]:
def classify_crop(crop_rgb):
    crop_input = preprocess_crop_for_classifier(crop_rgb, CLASSIFIER_IMG_SIZE)
    probs = classifier.predict(crop_input[None, ...], verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    pred_label = CLASS_NAMES[pred_idx]
    pred_conf = float(np.max(probs))
    return pred_label, pred_conf, probs

In [ ]:
def run_pipeline_on_image(image_path):
    image_path = Path(image_path)
    image_bgr = load_image_bgr(image_path)
    image_rgb = bgr_to_rgb(image_bgr)

    image_resized, scale, pad_left, pad_top, orig_h, orig_w = preprocess_for_detector(
        image_rgb,
        DETECTOR_TARGET_SIZE,
    )

    preds = detector.predict(image_resized[None, ...], verbose=0)

    num_det = int(preds["num_detections"][0]) if "num_detections" in preds else len(preds["boxes"][0])
    pred_boxes = np.asarray(preds["boxes"][0][:num_det], dtype=np.float32).reshape(-1, 4)
    pred_scores = np.asarray(preds["confidence"][0][:num_det], dtype=np.float32).reshape(-1)

    results = []

    for box_resized, det_score in zip(pred_boxes, pred_scores):
        if det_score < DETECTOR_CONFIDENCE_THRESHOLD:
            continue

        box_orig = map_box_to_original(
            box_resized,
            scale=scale,
            pad_left=pad_left,
            pad_top=pad_top,
            orig_h=orig_h,
            orig_w=orig_w,
        )

        box_crop = expand_box(box_orig, image_rgb.shape, margin=CROP_MARGIN)
        if box_crop is None:
            continue

        crop_rgb = crop_from_box(image_rgb, box_crop)
        if crop_rgb is None:
            continue

        pred_label, cls_conf, cls_probs = classify_crop(crop_rgb)

        results.append({
            "det_box_resized": box_resized,
            "det_box_orig": box_orig,
            "crop_box_orig": box_crop,
            "det_conf": float(det_score),
            "cls_label": pred_label,
            "cls_conf": cls_conf,
            "cls_probs": cls_probs,
            "crop_rgb": crop_rgb,
        })

    return {
        "image_path": str(image_path),
        "image_rgb": image_rgb,
        "results": results,
    }

In [ ]:
def show_pipeline_result(pipeline_output, show_crops=False):
    image_rgb = pipeline_output["image_rgb"]
    results = pipeline_output["results"]

    fig, ax = plt.subplots(1, figsize=(10, 10))
    ax.imshow(image_rgb)

    for r in results:
        x1, y1, x2, y2 = r["crop_box_orig"]

        det_conf = r["det_conf"]
        cls_conf = r["cls_conf"]

        if cls_conf >= 0.85:
            edge_color = "limegreen"
            face_color = "forestgreen"
        elif cls_conf >= 0.65:
            edge_color = "orange"
            face_color = "darkorange"
        else:
            edge_color = "red"
            face_color = "firebrick"

        rect = patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2.2,
            edgecolor=edge_color,
            facecolor="none",
        )
        ax.add_patch(rect)

        label_text = f"{r['cls_label']} | det {det_conf:.2f} | cls {cls_conf:.2f}"

        ax.text(
            x1,
            max(0, y1 - 8),
            label_text,
            color="white",
            fontsize=10,
            bbox=dict(facecolor=face_color, alpha=0.88, pad=2),
        )

    ax.set_title(Path(pipeline_output["image_path"]).name)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print(f"Detections kept: {len(results)}")
    for i, r in enumerate(results, start=1):
        print(
            f"{i:02d}: class={r['cls_label']} | det_conf={r['det_conf']:.4f} | "
            f"cls_conf={r['cls_conf']:.4f} | box={r['crop_box_orig'].tolist()}"
        )

    if show_crops and len(results) > 0:
        cols = 5
        rows = int(np.ceil(len(results) / cols))

        fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
        axes = np.array(axes).reshape(-1)

        for ax in axes[len(results):]:
            ax.axis("off")

        for ax, r in zip(axes, results):
            ax.imshow(r["crop_rgb"])
            ax.set_title(f"{r['cls_label']} ({r['cls_conf']:.2f})")
            ax.axis("off")

        plt.tight_layout()
        plt.show()

In [ ]:
output = run_pipeline_on_image(INFERENCE_IMAGE_PATH)
show_pipeline_result(output, show_crops=True)

In [ ]:
def run_pipeline_on_folder(folder_path, extensions=(".jpg", ".jpeg", ".png")):
    folder_path = Path(folder_path)
    image_paths = []
    for ext in extensions:
        image_paths.extend(folder_path.rglob(f"*{ext}"))
        image_paths.extend(folder_path.rglob(f"*{ext.upper()}"))

    image_paths = sorted(set(image_paths))
    print(f"Found {len(image_paths)} images.")

    outputs = []
    for image_path in image_paths:
        try:
            out = run_pipeline_on_image(image_path)
            outputs.append(out)
            print(f"Processed: {image_path.name} | detections={len(out['results'])}")
        except Exception as e:
            print(f"Failed: {image_path.name} | {e}")

    return outputs

In [ ]:
def export_pipeline_results(outputs, csv_path):
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "image",
            "det_index",
            "x1", "y1", "x2", "y2",
            "det_conf",
            "pred_class",
            "cls_conf",
             "threshold"
        ])

        for out in outputs:
            image_name = Path(out["image_path"]).name
            for i, r in enumerate(out["results"]):
                x1, y1, x2, y2 = r["crop_box_orig"]
                writer.writerow([
                    image_name,
                    i,
                    int(x1), int(y1), int(x2), int(y2),
                    round(r["det_conf"], 6),
                    r["cls_label"],
                    round(r["cls_conf"], 6),
                    DETECTOR_CONFIDENCE_THRESHOLD
                ])

    print("Saved:", csv_path)

In [ ]:
FOLDER_PATH = INFERENCE_IMAGE_PATH.parent  
outputs = run_pipeline_on_folder(FOLDER_PATH)
export_pipeline_results(outputs, INFERENCE_RESULTS_CSV)